
# PS2: Mini-batch SGD and CNN

Advanced Learning 2026/7.



For SUBMISSION:   

Please upload the complete and executed `ipynb` to your git repository. Verify that all of your output can be viewed directly from github, and provide a link to that git file below.

~~~
STUDENT ID: 319043287
~~~

~~~
STUDENT GIT LINK: https://github.com/Dubious14/52025---Adv.-computational-learning-Assigmants/tree/main/PS2
~~~
In Addition, don't forget to add your ID to the files, and upload to moodle the html version:    
  
`PS2_CNN_RNN_2026_ID_[319043287].html`   




---


Imports:

In [1]:
import numpy as np # You are allowed to use  only numpy.
import time, sys, cProfile
import matplotlib.pyplot as plt

Please import the objects in `srcPS2.py` file into the workspace

In [40]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [1]:
from srcPS2 import *

Note that now all of the network's objects from PS1 are loaded into the workspace.

**Alternatively**, you can upload srcPS2.py in the files tab found on the left hand side without needing to mount your drive, and import as is.

---

### 0. Loading data

As in PS1, you will evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.  



In [2]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


---

## PS2 Part 1: Mini-batch SGD



**Welcome back**.   






Mini-batch Stochastic Gradient Descent (Mini-batch SGD) is an optimization algorithm used to train machine learning models, particularly neural networks. It strikes a balance between two other methods:

    Batch Gradient Descent – Uses the entire dataset to compute gradients and update the model parameters.
        ✅ Accurate gradients
        ❌ Slow and memory-intensive for large datasets

    Stochastic Gradient Descent (SGD) – Uses a single data point to compute gradients for each update.
        ✅ Fast and efficient
        ❌ Noisy updates, which can lead to instability

  
**Mini-batch SGD**.

Mini-batch SGD computes gradients and updates the model parameters using small, randomly selected batches of data (typically 32, 64, or 128 samples).

    ✅ Faster than batch gradient descent
    ✅ Less noisy than pure SGD
    ✅ Efficient use of memory
    ✅ Enables vectorization for faster computation on GPUs

**How it Works**.

    Shuffle the dataset to avoid bias.

    Divide the data into small batches (e.g., 64 samples per batch).

    For each epoch, iterate through the mini-batches:
        Perform forward pass to calculate predictions.
        Compute the loss for the mini-batch.
        Perform backpropagation to compute gradients.
        Update model parameters using the gradients.

    Repeat until the model converges.

**Algorithm**

For each mini-batch $B={x_1,x_2,...,x_m}B={x_1​,x_2​,...,x_m​}$ of size $m$, update the weights $w$ as:    

$$
w=w−η⋅\frac{1}{m}∑_{i=1}^m ∇L(w,x_i)
$$

Where:   
*  $η$ = learning rate.
* $∇L(w,x_i)$ = gradient of the loss for sample $x_i$.   

Advantages

    Efficient and scalable – Works well with large datasets.
    Smooth convergence – Less noisy than SGD but faster than full batch training.
    Parallelization – Batches can be processed in parallel on GPUs.

Disadvantages

    Choosing batch size can affect performance. Small batches lead to noise, large batches consume more memory.
    Convergence may be slower than batch gradient descent for some problems.


### Mini-batch Questions.
**P1Q1**:  

Please answer the following questions providing statistical justifications and mathematical formulas when needed.

**P1Q1a:**   
Explain how the mini-batch size impacts convergence, model performance, and computational efficiency.




With a small mini-batch size, the gradient is calculated using only a subset of the data, so its direction might differ from the full gradient. The mini-batch gradient is:

$$
g_B(\theta)=\frac{1}{B}\sum_{i\in B}\nabla_\theta \ell_i(\theta)
$$

Under uniform random sampling, the mini-batch gradient is an unbiased estimate of the full gradient, but its variance depends on the batch size:

$$
\mathbb{E}[g_B(\theta)]=\nabla L(\theta),
\qquad
\operatorname{Cov}(g_B(\theta))\approx\frac{\Sigma(\theta)}{B}
$$

Here, $B$ is the mini-batch size and $\Sigma(\theta)$ is the covariance matrix of the individual gradients.

Therefore, smaller batches produce noisier gradients, which can lead to a zigzag path and slower or less stable convergence. I see it as walking in the dark with a small flashlight, where each step is based on the limited area we can see.

In terms of model performance, this randomness means that mini-batch SGD might reach different parameters than full-batch GD. However, the noise can sometimes help the model escape poor regions of the loss function and generalize better.

In terms of computational efficiency, smaller batches require less computation per update and allow more frequent weight updates, but their noisier gradients might require more updates to converge. Larger batches produce more stable gradients and can make better use of parallel computation, especially on GPUs, but each update is more computationally expensive. Therefore, the most efficient batch size depends on both the number of updates needed and the computational cost of each update.

**P1Q1b:**   
Why does mini-batch SGD require a learning rate schedule, and what are the common scheduling strategies?



At the first iterations, we are usually far from the optimal parameters, so a relatively large learning rate can help us make faster progress. As we approach the minimum, we want to take smaller steps to avoid overshooting it.

With full-batch GD, the gradient is calculated using the entire dataset. As we approach a smooth minimum, the gradient tends to become smaller, so the updates naturally become smaller even with a constant learning rate.

With mini-batch SGD, however, each gradient is calculated using only a subset of the data. The update rule is:

$$
\theta_{t+1}=\theta_t-\eta_t g_{B_t}(\theta_t)
$$

where $\eta_t$ is the learning rate and $g_{B_t}(\theta_t)$ is the gradient calculated from the current mini-batch.

Even at a minimum of the full training loss, where $\nabla L(\theta^*)=0$, the mini-batch gradient can still be nonzero:

$$
\mathbb{E}[g_B(\theta^*)]=0,
\qquad
\operatorname{Var}(g_B(\theta^*))>0
$$

This means that different mini-batches can keep pushing the parameters in different directions, causing them to "dance" around the minimum instead of settling there. A learning rate schedule helps reduce these fluctuations by gradually decreasing the size of the updates.

Common scheduling strategies are:

1. **Step Decay** – Decrease the learning rate by a constant ratio after a predetermined number of steps.
2. **Exponential Decay** – Gradually decrease the learning rate exponentially.
3. **Linear Decay** – Gradually decrease the learning rate at a constant rate.
4. **Reduce on Plateau** – Decrease the learning rate when a metric, such as validation loss, stops improving for a certain number of epochs.

**P1Q1c:**   
What role does batch normalization play in mini-batch SGD, and why is it  essential for deep neural networks?



Batch normalization normalizes the activations using the mean and variance of each mini-batch, helping keep them on a more consistent scale. This makes optimization easier, since poorly scaled activations can lead to very large or small gradients, making it difficult to choose an appropriate learning rate.

In mini-batch SGD, the normalization statistics are calculated from a different subset of observations in each iteration. This introduces additional randomness, which can act as regularization and sometimes improve generalization.

Batch normalization is especially useful in deep neural networks, where gradient problems can accumulate across many layers. By improving gradient behavior and making optimization more stable, it often allows larger learning rates and faster convergence.

**P1Q1d:**   
How does mini-batch SGD differ in performance compared to full-batch gradient descent when dealing with non-convex optimization problems?



In non-convex problems, the loss function can have multiple local minima and saddle points. Full-batch GD follows the exact gradient, so its convergence is more stable. However, at a local minimum or saddle point, the gradient is zero, so GD cannot move away if it reaches that point exactly. Mini-batch SGD introduces randomness because each gradient is calculated using only a subset of the data. Therefore, even when the full gradient is zero, the mini-batch gradient might not be, allowing the model to escape saddle points or some poor local minima and potentially find a better solution.

**P1Q1e:**   
How does mini-batch SGD perform when training on imbalanced datasets, and what strategies can mitigate its challenges?



In imbalanced datasets, the majority class contributes more to the loss, which can cause the model to perform poorly on the minority class. With mini-batch SGD, the number of minority-class observations varies between batches.

Let $p$ be the proportion of minority-class observations and $B$ the mini-batch size. Under independent random sampling, the number of minority-class observations in a mini-batch follows:

$$
K\sim\operatorname{Binomial}(B,p)
$$

Therefore:

$$
\mathbb{E}[K]=Bp,
\qquad
P(K=0)=(1-p)^B
$$

When the minority class is rare (very small $p$) relative to the batch size, some batches might not contain any observations from it. This makes its contribution to the gradient updates less consistent, increasing gradient variability and making learning less stable. Although these differences average out in expectation, the weights are updated after every batch, so the updates don't necessarily cancel each other out.

To address this, we could use stratified mini-batches to ensure consistent class representation, oversample the minority class, or use a weighted loss function to give more importance to minority-class observations.

**P1Q1f:**   
Explain why larger mini-batch sizes are preferred during inference but not necessarily during training.

During inference, we only perform forward propagation, so larger batches allow us to process more observations in parallel, making better use of the GPU and reducing computation time per observation. During training, we also need to store intermediate activations for backpropagation. Therefore, for the same batch size, we need more memory. larger batches reduce the randomness of the gradient estimates, but this randomness can sometimes help the model find better solutions and generalize better.

### Applying mini-batch SGD

Preparing the data:

In [3]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

P1Q2a:  

Add a new function to the class `MyNetwork` named `fit_mini_batch`. This function should be similar to `fit`, just with a mini-batch SGD implementation. You can also make other code changes to the scripts in `srcPS2` if you think they are needed. In addition, add a suitable learning rate scheduler of your choice.


YOU SHOULD (a) add the missing script to the srcPS2 code, and (b) make a copy of the function  below for review:

In [4]:
class ExponentialDecayScheduler:
    def __init__(self, initial_lr, decay_rate):
        self.initial_lr = initial_lr
        self.decay_rate = decay_rate

    def get_lr(self, epoch):
        return self.initial_lr * (self.decay_rate ** epoch)

def fit_mini_batch(self, x_train, y_train, batch_size, epochs, learning_rate):
    """
    Trains the network using mini-batch gradient descent.

    Parameters:
      x_train : np.array
          Training data.
      y_train : np.array
          Training labels.
      batch_size : int
          Size of each mini-batch.
      epochs : int
          Number of training epochs.
      learning_rate : float
          Learning rate for updating the weights.
    """
    samples = len(x_train)
    num_batches = int(np.ceil(samples / batch_size))
    print(f'Training on {samples} samples, divided into {num_batches} batches')
    scheduler = ExponentialDecayScheduler(learning_rate, 0.95)

    for i in range(epochs):
        current_lr = scheduler.get_lr(i)
        epoch_err = 0

        indices = np.random.permutation(samples)
        x_train = x_train[indices]
        y_train = y_train[indices]

        for batch in range(num_batches):
            start = batch * batch_size
            x_batch = x_train[start:start + batch_size, 0, :]
            y_batch = y_train[start:start + batch_size]

            output = x_batch
            for layer in self.layers:
                output = layer.forward_propagation(output)

            batch_grad = self.loss_grad(y_batch, output)
            batch_err = self.loss(y_batch, output)
            epoch_err += batch_err * len(x_batch)

            for layer in reversed(self.layers):
                batch_grad = layer.backward_propagation(batch_grad, current_lr)

        epoch_err /= samples
        print(f'Training epoch: {i+1}, Learning rate: {current_lr:.6f}, Error: {epoch_err:.6f}')



    Test the two networks below to compare the differences between GD and mini-batch SGD:

In [ ]:
#### GD implementaiton ####

# Network Architecture
netGD = MyNetwork()

netGD.add(Affine_Layer(28*28, 128))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(128, 64))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(64, 10))
netGD.add(ActivationLayer(tanh, tanh_grad))

netGD.use_loss(mse, mse_grad)

epoch_num = 10
lr = 0.05
t1 = time.time()
netGD.fit(x_train[:10000], y_train[:10000], epochs=epoch_num, learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")

Training on 10,000 samples:
Training epoch 1/10   error=0.041134
Training epoch 2/10   error=0.023806
Training epoch 3/10   error=0.018754
Training epoch 4/10   error=0.015853
Training epoch 5/10   error=0.013817
Training epoch 6/10   error=0.012270
Training epoch 7/10   error=0.011031
Training epoch 8/10   error=0.009999
Training epoch 9/10   error=0.009122
Training epoch 10/10   error=0.008371
Total process time: 30.078


In [ ]:
output_GD = netGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0128, Accuracy: 9379/10000 (94%)



In [ ]:
#### mini-batch SGD implementaiton ####

# Network Architecture
netMiniGD = MyNetwork()

netMiniGD.add(Affine_Layer(28*28, 128))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(128, 64))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(64, 10))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))

netMiniGD.use_loss(mse, mse_grad)

epoch_num = 10
lr_sched = 0.05 # add a learning rate scheduler of your choice here
t2 = time.time()
netMiniGD.fit_mini_batch(x_train[:10000], y_train[:10000], batch_size=128, epochs=epoch_num, learning_rate=lr_sched)
print(f"Total process time: {round(time.time() - t2,3)}")


Training on 10000 samples, divided into 79 batches
Training epoch: 1, Learning rate: 0.050000, Error: 0.129348
Training epoch: 2, Learning rate: 0.047500, Error: 0.087245
Training epoch: 3, Learning rate: 0.045125, Error: 0.074701
Training epoch: 4, Learning rate: 0.042869, Error: 0.068175
Training epoch: 5, Learning rate: 0.040725, Error: 0.064051
Training epoch: 6, Learning rate: 0.038689, Error: 0.061185
Training epoch: 7, Learning rate: 0.036755, Error: 0.059048
Training epoch: 8, Learning rate: 0.034917, Error: 0.057377
Training epoch: 9, Learning rate: 0.033171, Error: 0.056019
Training epoch: 10, Learning rate: 0.031512, Error: 0.054905
Total process time: 4.755


In [ ]:
outputMiniGSD = netMiniGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0553, Accuracy: 7662/10000 (77%)



P1Q2b:   
Please answer the following:
* Which implementation was faster?
* with lower training error?
* with better test accuracy?    

and explain how these results fit your answers in Q1A.

### P1Q2b: Methods Comparsion
The mini-batch SGD was much faster with total process time of 4.755 seconds, compared to the 30.078 seconds of the per sample SGD, but achieved worse training error (0.054905 vs. 0.008371) and worse test accuracy (77% vs. 94%).

One possible explanation is the difference in the number of parameter updates. With per-sample SGD, the parameters are updated after every observation, resulting in 10,000 × 10 = 100,000 updates. With mini-batch SGD, the parameters are updated after every batch, resulting in only 79 × 10 = 790 updates.

As discussed in Q1a, smaller batches produce noisier gradient estimates, while larger batches produce more stable gradients and benefit from parallel computation. This explains why mini-batch SGD was much faster. However, despite having noisier gradients, per-sample SGD performed better in this experiment, possibly because it had many more opportunities to update the parameters during the same 10 epochs.

In addition, I used the same initial learning rate of 0.05 for both implementations, but added an exponential learning rate schedule to mini-batch SGD, decreasing the learning rate by a factor of 0.95 after each epoch. The combination of fewer updates and a decreasing learning rate may have prevented the mini-batch model from also converging within 10 epochs.

## PS2 Part 2: CNN



A Convolutional Neural Network (CNN) is a type of a deep neural network that can perform well at image recognition and analysis tasks.


> In this problem set we will add CNN compatibility to our numpy-based neural network from PS 1.

The main components needed to implement a CNN architecture:  

**Convolution layer:** In the convolutional layers, filters are applied to the image to identify patterns and features. Imagine a sliding window that moves across the image, detecting edges, shapes, and other building blocks.  

**Pooling layer:** Pooling layers downsample the data, reducing its complexity and computational cost. This helps prevent overfitting, where the model memorizes training data instead of learning generalizable patterns.  

**Reshape layer:**
Reshaping the output of the convolution and pooling layers into the fully connected layer.   


A USEFUL HINT:

*Keeping account of the input and output shapes along the CNN network is crucial for the model to work as intended*



Useful resource:  
* [CS231](https://cs231n.github.io/convolutional-networks/).
* [Convolutional Neural Network From Scratch](https://www.kaggle.com/code/lusfernandotorres/convolutional-neural-network-from-scratch).
* [CNN with PyTorch](https://www.kaggle.com/code/sdelecourt/cnn-with-pytorch-for-mnist)

#### Data transformations





(Here I added a reshape that I originally intended for you to apply at model time).

In [5]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [6]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.astype('float32')
x_train /= 255
x_train = x_train.reshape(x_train.shape[0], 1, x_train.shape[1],x_train.shape[2])
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
y_train = y_train.reshape(y_train.shape[0], 1, y_train.shape[1])

# same for test data : 10000 samples
x_test = x_test.astype('float32')
x_test /= 255
x_test = x_test.reshape(x_test.shape[0],1, x_test.shape[1],x_test.shape[2])
y_test = to_categorical(y_test)
y_test = y_test.reshape(y_test.shape[0],1, y_test.shape[1])

In [7]:
print("x_train shape",x_train.shape)
print("x_test shape",x_test.shape)
print("y_train shape",y_train.shape)
print("y_test shape",y_test.shape)

x_train shape (60000, 1, 28, 28)
x_test shape (10000, 1, 28, 28)
y_train shape (60000, 1, 10)
y_test shape (10000, 1, 10)


Unlike the multilayer perceptron (MLP) architecture, the input to the convolution net is not flattened. In our case, the size of a single `MNIST` datapoint is (1,28,28). The first dimension is the number of channels (e.g. RGB  for color images). During training the inputs and outputs often  change shape as they pass through the convolution, pooling, and the other layers of the network.

---

### A. Convolution Layer

Here you are asked to implement a basic Convolution layer with the following components:



1.   Layer initialization - initialize the layer's filters using Normalized Xavier/Hu weights
2.   A generator function that yields all the available sliding windows in the input image.
3.   Forward propogation for the convolution layer
4.   Backward propogation for the convolution layer

[CS231](https://cs231n.github.io/convolutional-networks/#conv) provides a great overview both mathematically and visually,   
and [CS230](https://cs230.stanford.edu/section/4/) provides a good overview about Xavier initialization.




P2Q1a:  (10pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.    
You can deviate slightly from the recommended structure below, but please remain true to the class structure and compatibility with the rest of the network classes and functions. The idea here is to write YOUR OWN version of CNN, so please try not to google the answers.





In [8]:
### MISSING: SOLUTION REQUIRED IN THIS CODE BLOCK

class Convolution:
    # convolution layer using num_filters x num_filters filters.
    # size is the length/width of the filter window with shape (size x size)
    # In this simple implementation, there is no padding.

    def __init__(self, input_size, output_size=3, size=3, activation=None):
        self.stride = 1 # for simplicity, we keep the stride=1.
        self.size = size
        self.activation = activation
        self.last_input = None

        # Initialize the Convolution layer filters
        ## SOLUTION REQUIRED!!
        fan_in = input_size * size ** 2
        fan_out = output_size * size ** 2

        self.filters = np.random.randn(output_size, input_size , size, size) * np.sqrt(2 / (fan_in + fan_out))


    def patch_generator(self, image):
        # This generator (using yield) returns all the  possible patches of the input image
        # of shape  (h,w)  for each channel c, together with the relevant y,x coordinates of the patch.
        # input_data/image must be shaped as (c,y,x), where c is the channel/filter
        ch,h,w = image.shape

        for y in range(0, h - self.size + 1):
            for x in range(0, w - self.size + 1):
                patch = image[:, y:y + self.size, x: x+ self.size]
                yield patch, y, x


    def forward_propagation(self, image):
        # Calculate the convolution forward pass using the filters together with a
        # user specified activation function.

        # If this is the first convolution layer, the input image is size (h,w) and must be expanded to (c,h,w).
        if len(image.shape) == 2:
            image = np.expand_dims(image, axis=0)
        # keep track of last input for later backward propagation
        self.last_input = image
        # Initializing the output array:
        ## SOLUTION REQUIRED!!
        input_dimension = image.shape[1]
        output_dimension = (input_dimension - self.size) // self.stride + 1

        # compute output dimensions after the convolution layer
        output = np.zeros((self.filters.shape[0], output_dimension, output_dimension))

        # forward pass on each filter
        for f in range(self.filters.shape[0]):
            for patch, y, x in self.patch_generator(image):
                ## SOLUTION REQUIRED!!
                output[f, y, x] = np.sum(self.filters[f] * patch)

        # Applying activation function (this is a small cheat/simplification
        #  because the activation should be a separate layer in itself.)
        if self.activation is not None:
            output = self.activation.forward_propagation(output)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        # Calculate the SGD in-gradients of the filters and the layer.
        # Update only the filters gradients.

        # back propagate through activation (again, a bit of a cheat)
        if self.activation is not None:
            grad_out = self.activation.backward_propagation(grad_out, learning_rate)

        ## SOLUTION REQUIRED!!
        grad_in = np.zeros(self.last_input.shape)  # loss gradient of the input
        grad_filter = np.zeros(self.filters.shape) # loss gradient of filter


        for f in range(self.filters.shape[0]):
            for patch, y, x in self.patch_generator(self.last_input):
                ## SOLUTION REQUIRED!!
                grad_filter[f] += patch * grad_out[f, y, x]
                grad_in[:, y:y + self.size, x:x + self.size] += self.filters[f] * grad_out[f, y, x]

        ## SOLUTION REQUIRED!!
        self.filters -= learning_rate * grad_filter # update filters using SGD

        return grad_in    # return the loss gradient for this layer's inputs


### B. Max-pooling Layer

As with the convolution layer, here you are asked to implement a basic max-pooling layer with the following components:


1.   A generator function that yields all the available sliding windows in the input image. You can reuse the generator from (1).
2.   Forward propogation for the max-pooling layer
3.   Backward propogation for the max-pooling layer

The pooling layer is responsible for reducing the dimensionality of the input. As with the convolution layer, the pooling layer slides a filter across the entire image input to calculate the output. Note that this layer does not have any weights.  As the filter slides through the input image, it selects the pixel with the maximum value to populate the output array.

P2Q1b: (10pts).       

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [9]:
### MISSING CODE IN THIS BLOCK

class MaxPool:
    def __init__(self,  size=2):
        # max pooling layer

        self.last_input = None
        self.stride = 1
        self.size = size

    def patch_generator(self, image):
        # returns a generator (using yield) of all possible patches of size size x size
        # input_data/image must be shaped as (c,y,x)
        ch, h, w = image.shape

        for c in range(ch):
          for y in range(0, h - self.size + 1):
              for x in range(0, w -self.size + 1):
                  patch = image[c, y: y + self.size, x: x + self.size]
                  yield patch, c, y, x

    def forward_propagation(self, image):
        # keep track of last input for later backward propagation
        self.last_input = image

        # compute output dimensions after the max pooling
        num_channels, h_prev, w_prev = image.shape
        ## SOLUTION REQUIRED!!
        h_new = (h_prev - self.size) // self.stride + 1
        w_new = (w_prev - self.size) // self.stride + 1
        # hold the values of the max pooling
        output = np.zeros((num_channels, h_new, w_new))

        # Calculate the max pool on all the image patches using the generator:
        # CODE MISSING
        for patch, c, y, x in self.patch_generator(image):
            output[c, y, x] = np.max(patch)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        # Calculate the gradients of the last input image
        grad_in = np.zeros(self.last_input.shape)

        # Using the generator, go over all the channels and calcualte the max-pooling input gradients
        # for c in range(num_channels):
        for patch, c, y, x in self.patch_generator(self.last_input):
            max_y, max_x = np.unravel_index(np.nanargmax(patch), patch.shape)
            ## SOLUTION REQUIRED!!
            grad_in[c, y + max_y, x + max_x] += grad_out[c, y, x]

        return grad_in


### C. Reshape Layer

A reshape layer in a neural network is a transformation layer that doesn't perform any computations itself (no weights).   
Its purpose is to simply change the dimensionality (shape) of the data flowing through the network.

Here's how reshape layers work:

You define a target shape for the output of the reshape layer.   
This target shape specifies the number of elements along each dimension (width, height, channels, etc.) for the new data format.

P2Q1c:  (5pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [10]:
### MISSING CODE IN THIS BLOCK ####
class Reshape:
    def __init__(self,input_shape,output_shape):
        self.input_shape=input_shape
        self.output_shape=output_shape
    def forward_propagation(self,input_data):
        ## SOLUTION REQUIRED!!
        return np.reshape(input_data, self.output_shape)

    def backward_propagation(self,output_gradient,learning_rate):
        # you can ignore the learning_rate
        ## SOLUTION REQUIRED!!
        return np.reshape(output_gradient, self.input_shape)


### Testing Your Neural Network

### Defining our main neural network architecture

Define your CNN network's architecture:

* A convolution layer with some filters `(f)` and a window size of `(w x w)`, with a `tanh` activation function.
* A max pooling layer of a window size of `(p x p)`.  
* First affine layer that takes your input and outputs 128 nodes
* `tanh` activation layer following the first affine layer
* Second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh` activation layer following the second affine layer
* Third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh` activation layer following the last affine layer


You can compute the shapes of the convolution and max-pool layers' outputs  as a function of the input shape, the convolution sliding window size, and the number of filters for applied. ( the stride and padding are usually also used to calculate the output shapes, but in this simple implementation we set the stride to 1 with zero padding.).

P2Q2a: (5pts).        
Calculate the input and output shapes of each layer in the network below, up until the first affine layer (including).  (mathematically) explain your answer.  

### P2Q2a: Mathematical Derivation of Layer Shapes

Let the input image have shape:

$$
X \in \mathbb{R}^{C \times H \times W}
$$

where $C$ is the number of input channels, and $H,W$ are the image height and width.

We assume stride $s=1$ and no padding for both convolution and max pooling.

**1. Convolution layer**

The convolution uses $f$ filters, each with a spatial window of size $w \times w$.

The output spatial dimensions are:

$$
H_{\text{conv}} = H-w+1, \qquad W_{\text{conv}} = W-w+1
$$

Since each filter produces one output feature map, the output shape is:

$$
\boxed{(f,\ H-w+1,\ W-w+1)}
$$

The $\tanh$ activation does not change the shape.

**2. Max pooling layer**

Max pooling uses a $p \times p$ window and operates independently on each of the $f$ feature maps.

The output spatial dimensions are:

$$
H_{\text{pool}} = H_{\text{conv}}-p+1 = H-w-p+2
$$

$$
W_{\text{pool}} = W_{\text{conv}}-p+1 = W-w-p+2
$$

Therefore, the output shape is:

$$
\boxed{(f,\ H-w-p+2,\ W-w-p+2)}
$$

**3. Flattening (Reshape)**

Before the first affine layer, the output of MaxPool is flattened into a vector.

The number of features is:

$$
D = f(H-w-p+2)(W-w-p+2)
$$

Thus, the shape changes from:

$$
(f,\ H-w-p+2,\ W-w-p+2)
\longrightarrow (1,D)
$$

**4. First affine layer**

The first affine layer receives $D$ input features and produces 128 output neurons.

Therefore:

$$
\boxed{(1,D) \longrightarrow (1,128)}
$$

where:

$$
\boxed{D=f(H-w-p+2)(W-w-p+2)}
$$

P2Q2b:  (5pts).   

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the network architecture works properly.  





In [14]:
#### SOLUTION REQUIRED IN THIS BLOCK (in reshape parameters) ####

# Network Architecture
CNNnet = MyNetwork()
num_classes = 10
# Reshape sizes:
## SOLUTION REQUIRED!!
c = 12
x_new = (x_train.shape[3] - 3 - 3 + 2)
y_new = (x_train.shape[2] - 3 - 3 + 2)

# Adding convolution and max-pool layers:
CNNnet.add(Convolution(input_size=x_train.shape[1], output_size=c, size=3,
    activation=ActivationLayer(relu, relu_grad)
))
CNNnet.add(MaxPool(size=3))

# Reshaping for first affine layer:
CNNnet.add(Reshape((c, y_new, x_new), (1, c * y_new * x_new)))

# Adding affine layers and activation functions:
CNNnet.add(Affine_Layer(c * y_new * x_new, 128))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(128, 64))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(64, num_classes))
CNNnet.add(ActivationLayer(tanh, tanh_grad))

# CNNnet.add(ActivationLayer(relu, relu_grad))
# CNNnet.add(ActivationLayer(tanh, tanh_grad))
# CNNnet.add(ActivationLayer(sigmoid, sigmoid_grad))

### Training!

There is no need to train the model on the entire dataset (you'l see why now...). Try to prove that your network works as expected by showing that your error decreases dramatically even on a small sample (say, around 1k).

In [16]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# Training will be pretty slow if we update at each iteration on 60000 samples...]
CNNnet.use_loss(mse, mse_grad)
epoch_num = 3
lr = 0.1
t1 = time.time()
CNNnet.fit(x_train[:1000], y_train[:1000], epochs=epoch_num,
           learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")


Training on 1,000 samples:
Training epoch 1/3   error=0.020929
Training epoch 2/3   error=0.017679
Training epoch 3/3   error=0.014941
Total process time: 902.485


Hurray - if you got to this point, your CNN model works, but is EXTREMELY slow.   
Let's check why using a cProfiler:

In [17]:
CNNnet.prof(x_train[:100], y_train[:100],epochs=4,learning_rate=lr)


Training on 100 samples:
Training epoch 1/4   error=0.012705
Training epoch 2/4   error=0.008413
Training epoch 3/4   error=0.006602
Training epoch 4/4   error=0.005484
         117244558 function calls (117244505 primitive calls) in 166.140 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      400    0.004    0.000    0.011    0.000 2849307609.py:10(backward_propagation)
      400    0.003    0.000    0.011    0.000 2849307609.py:6(forward_propagation)
  5530400    3.790    0.000    3.790    0.000 3574549612.py:11(patch_generator)
      400    3.086    0.008   15.632    0.039 3574549612.py:22(forward_propagation)
      400   14.312    0.036   85.404    0.214 3574549612.py:40(backward_propagation)
  6499200    3.545    0.000    3.545    0.000 50112235.py:22(patch_generator)
      400   10.443    0.026   26.424    0.066 50112235.py:34(forward_propagation)
      400   23.504    0.059   25.101    0.063 50112235.py:63(backward_

P2Q2c:  (5pts).     
Why do you think our CNN implementation is slow? Your answer should relate to the  approximate `O(n)` time/computational complexity .



The approximate computational complexity of our convolutional layer per training epoch is:

$$
\boxed{O(N \cdot F \cdot C \cdot K^2 \cdot H_{\text{out}} \cdot W_{\text{out}})}
$$

Where:

- $N$: Number of training images.
- $F$: Number of output filters (12 in our implementation).
- $C$: Number of input channels (1 for MNIST).
- $K^2$: Number of weights in each 2D kernel ($3 \times 3 = 9$).
- $H_{\text{out}}W_{\text{out}}$: Number of spatial positions where each filter is applied ($26 \times 26 = 676$).

For each image, we apply all $F$ filters at every output position. Each application requires approximately $CK^2$ multiplications and additions. Backpropagation has the same asymptotic complexity, since it also iterates over all filters and patches to calculate weight and input gradients.

Our code processes each patch separately using nested Python loops, repeatedly calling NumPy functions for very small operations. Each call introduces overhead from Python execution, function dispatch, and array handling.

The profiling results support this explanation: training on just 100 samples for 4 epochs required approximately 117 million function calls and 166 seconds. In particular, np.nanargmax was called approximately 2.76 million times, with a cumulative execution time of 76.2 seconds, highlighting the overhead of repeatedly processing individual pooling patches.

### Evaluation

Exciting! Now is the time to test your model.     


    May the gradients be always in your favor.

P2Q2b (cont'd):

**Try to limit the number of epochs and the size of the trained data, while still achieving an accuracy score of >85%.**

In [18]:
output = CNNnet.predict(x_test[:500] ,y_test[:500] )


Test set: Avg. loss: 0.0255, Accuracy: 460/500 (92%)



### Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

**Prepare the data as tensors using PyTorch DataLoader:**

In [33]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# # training data : 60000 samples
# # reshape and normalize input data
# x_train = x_train.reshape(x_train.shape[0], 1, 28*28) not needed
x_train = x_train.astype('float32')
x_train /= 255
# # One-hot encoding of the output.
# # Currently a number in range [0,9]; Change into a vector of size 10
# # e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# # same for test data : 10000 samples
# x_test = x_test.reshape(x_test.shape[0], 1, 28*28) not needed
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

In [34]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchCNN` class with an identical architecture you used in your home-made network.  
Hint: use `x = x.view()` to handle reshapes.

P2Q3a: (5pts).   

In [35]:
#### SOLUTION REQUIRED  IN THIS BLOCK####

class PyTorchCNN(nn.Module):
    def __init__(self):
        super(PyTorchCNN, self).__init__()

        input_size = c * y_new * x_new
        num_classes = y_train.shape[-1]

        self.conv = nn.Conv2d(1, c, kernel_size=3, stride=1,
                              padding=0, bias=False)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=1)
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.view(-1, 1, *x.shape[-2:])
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = torch.tanh(self.fc3(x))

        return x

In [36]:
# Train the model
num_epochs = 10
pt_learning_rate = 0.01
pt_network = PyTorchCNN()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()
t2 = time.time()

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        # labels = labels.view(64,1,10)
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

  print(f"Total process time: {round(time.time() - t2,3)}")


Epoch [1/10], Step [500/938], Loss: 0.0103
Epoch [2/10], Step [500/938], Loss: 0.0057
Epoch [3/10], Step [500/938], Loss: 0.0100
Epoch [4/10], Step [500/938], Loss: 0.0037
Epoch [5/10], Step [500/938], Loss: 0.0050
Epoch [6/10], Step [500/938], Loss: 0.0066
Epoch [7/10], Step [500/938], Loss: 0.0015
Epoch [8/10], Step [500/938], Loss: 0.0081
Epoch [9/10], Step [500/938], Loss: 0.0076
Epoch [10/10], Step [500/938], Loss: 0.0058
Total process time: 228.898


Evaluation:

In [41]:
pt_network.eval()
test_losses = []
test_loss = 0.0
correct = 0

with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)

        # MSELoss returns the mean loss for each batch
        test_loss += criterion(output, target).item() * data.size(0)

        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.argmax(dim=1, keepdim=True)).sum().item()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)

print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
    test_loss, correct, len(test_loader.dataset),
    100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0089, Accuracy: 9576/10000 (96%)



P2Q3b (5pts).   

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

### P2Q3b: Model Comparison and Improvements

**1. Which model performed better? Why?**

The PyTorch CNN achieved better test accuracy (96% vs. 92%). This is likely due in part to its more effective optimization using Adam and mini-batch training, as well as training for more epochs on more data. The comparison is not fully controlled: the NumPy CNN was trained on 1,000 samples for 3 epochs, while PyTorch used 60,000 samples for 10 epochs.

**2. Which model performed faster? Why?**

The PyTorch CNN was significantly faster, completing 10 epochs on 60,000 training samples in 228.9 seconds, compared to 902.5 seconds for the NumPy CNN, which trained for only 3 epochs on 1,000 samples. PyTorch was approximately 4 times faster overall, despite processing 200 times more training examples.

The main reason is that our NumPy implementation processes each convolution and pooling patch separately using nested Python loops. PyTorch uses optimized, compiled, and vectorized operations that process many calculations efficiently in parallel, while also supporting mini-batch training.

**3. What would you change in your network's architecture?**

I would increase the MaxPool stride from 1 to 2, reducing the spatial dimensions and the number of parameters in the first affine layer. I would also add another convolutional layer to learn more complex features. Finally, I would remove the final tanh activation and use logits with softmax and cross-entropy loss instead of MSE.

**4. What would you change in your model's solution algorithm?**

I would replace the per-sample SGD training with mini-batch optimization using Adam, which can provide more efficient and stable updates. I would also vectorize the convolution and pooling operations instead of processing each patch separately through Python loops. Additionally, I would use a validation set for early stopping and learning-rate adjustments to improve convergence and reduce unnecessary training.


In [39]:
!jupyter nbconvert --to html PS2_Part1_CNN_2026_ID_319043287.ipynb

[NbConvertApp] Converting notebook PS2_Part1_CNN_2026_ID_319043287.ipynb to html
[NbConvertApp] Writing 458210 bytes to PS2_Part1_CNN_2026_ID_319043287.html
